ALPHA_VANTAGE_API 무료 플랜 제한 사항
- 일일 호출 제한 25회
- 최근 100 거래일의 데이터만 받을 수 있음



Tiingo라는 대체 API 존재 (D19.ipynb에서 코드 확인 가능)
- Tiingo 무료 플랜 사용 시 시간당 50회
- 일 1,000회 요청이 가능
- 월별 고유 심볼은 최대 500
- 월 1GB bandwidth 제한
- 1회 요청으로 원하는 기간의 데이터를 한번에 받을 수 있음
- 원주가와 수정주를 한번에 제공

In [13]:
import requests
import pandas as pd
import os

API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")

SYMBOL = "AAPL"

ALPHAVANTAGE_URL = "https://www.alphavantage.co/query"

params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": SYMBOL,
    "outputsize": "compact",   # 무료: 최근 100 거래일
    "apikey": API_KEY,
}

# "outputsize":"full"로 25년+ 전체 데이터를 받는 기능은 Premium

response = requests.get(ALPHAVANTAGE_URL, params=params)
print(response.url)

response.raise_for_status()

data = response.json()

data

https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=AAPL&outputsize=compact&apikey=OSNDPAM8M304CM80


{'Meta Data': {'1. Information': 'Daily Prices (open, high, low, close) and Volumes',
  '2. Symbol': 'AAPL',
  '3. Last Refreshed': '2026-10-12',
  '4. Output Size': 'Compact',
  '5. Time Zone': 'US/Eastern'},
 'Time Series (Daily)': {'2026-10-12': {'1. open': '338.2150',
   '2. high': '342.6300',
   '3. low': '337.6000',
   '4. close': '340.9700',
   '5. volume': '25900599'},
  '2026-10-09': {'1. open': '331.6950',
   '2. high': '338.6100',
   '3. low': '330.7000',
   '4. close': '336.6400',
   '5. volume': '37881199'},
  '2026-10-08': {'1. open': '336.8150',
   '2. high': '341.5700',
   '3. low': '335.9000',
   '4. close': '340.4200',
   '5. volume': '35332449'},
  '2026-10-07': {'1. open': '336.9600',
   '2. high': '338.6700',
   '3. low': '332.7800',
   '4. close': '336.6700',
   '5. volume': '34147860'},
  '2026-10-06': {'1. open': '332.2800',
   '2. high': '334.3800',
   '3. low': '330.6200',
   '4. close': '333.6300',
   '5. volume': '30449005'},
  '2026-10-05': {'1. open': '332

In [16]:
meta_data = data['Meta Data']
meta_df = pd.DataFrame.from_dict(meta_data, orient='index')
meta_df

,0
1. Information,"Daily Prices (open, high, low, close) and Volumes"
2. Symbol,AAPL
3. Last Refreshed,2026-10-12
4. Output Size,Compact
5. Time Zone,US/Eastern


In [11]:
daily = data["Time Series (Daily)"]

df = (
    pd.DataFrame.from_dict(daily, orient="index")
    .rename(
        columns={
            "1. open": "open",
            "2. high": "high",
            "3. low": "low",
            "4. close": "close",
            "5. volume": "volume",
        }
    )
    .rename_axis("date")
    .reset_index()
)

df["date"] = pd.to_datetime(df["date"])

for col in ["open", "high", "low", "close"]:
    df[col] = pd.to_numeric(df[col])

df["volume"] = pd.to_numeric(df["volume"])

df

,date,open,high,low,close,volume
0,2026-10-12,338.215,342.63,337.60,340.97,25900599
1,2026-10-09,331.695,338.61,330.70,336.64,37881199
2,2026-10-08,336.815,341.57,335.90,340.42,35332449
3,2026-10-07,336.960,338.67,332.78,336.67,34147860
4,2026-10-06,332.280,334.38,330.62,333.63,30449005
...,...,...,...,...,...,...
95,2026-05-27,308.330,313.26,308.30,310.85,50430919
96,2026-05-26,309.560,311.82,307.67,308.33,48000493
97,2026-05-22,306.120,311.40,305.84,308.82,43670223
98,2026-05-21,301.055,305.54,300.40,304.99,42965126


In [6]:
def get_daily_stock_price(company, start_date, end_date):
    API_TOKEN = os.getenv('TIINGO_API_TOKEN')

    ticker = company
    start_date = start_date
    end_date = end_date

    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"

    params = {
        "startDate": start_date,
        "endDate": end_date,
        "token": API_TOKEN,
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame(data)
    raw_df = df[
        [
            "date",
            "open",
            "high",
            "low",
            "close",
            "volume",
        ]
    ].copy()

    raw_df["date"] = pd.to_datetime(raw_df["date"]).dt.date

    for col in ["open", "high", "low", "close"]:
        raw_df[col] = pd.to_numeric(raw_df[col])

    raw_df["volume"] = pd.to_numeric(raw_df["volume"])
    
    
    return raw_df

In [7]:
apple = get_daily_stock_price(
    company="AAPL",
    start_date = "2025-10-01",
    end_date = "2025-10-01"
    )
apple

,date,open,high,low,close,volume
0,2025-10-01,255.04,258.79,254.93,255.45,48713940
